In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings
from langchain_groq import ChatGroq


C:\Users\tembh\AppData\Local\Temp\ipykernel_4204\3444653110.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [3]:
pdf_object= PyPDFLoader("../Data/iot.pdf")
pdf=pdf_object.load()

In [4]:
splitter=RecursiveCharacterTextSplitter(chunk_size=400,chunk_overlap=100)
splitted_data=splitter.split_documents(documents=pdf)


In [10]:
vector_store=InMemoryVectorStore.from_documents(
    documents=splitted_data,
    embedding=OllamaEmbeddings(model="qwen3-embedding:8b")
)

In [6]:
@tool
def get_context(query:str):
    """
    This is a tool which will fetch the context of questions """
    data=vector_store.similarity_search(query=query)
    context=""
    for doc in data:
        context= context + doc.page_content
    return context

In [12]:
agent=create_agent(
    model=ChatGroq(model="openai/gpt-oss-20b"),
    tools=[get_context],
    system_prompt="You are an agent which uses tools to search for data required to answer question and if you dont get the answer in the fetched context reply i dont know or its out of context , never use pretrained data "
)

In [8]:
query="who is PM of india"

In [11]:
result=agent.invoke({"messages":[{"role":"user","content":query}]})

In [13]:
print(result["messages"][-1].content)

I’m sorry, but I don’t have that information in the fetched context.
